# NIST TN 1822 — Verif.2.3: Movement around a corner

20 agents start at one end of an L-shaped corridor and exit through the other. No boundary penetration should occur. Geometry follows the IMO/RiMEA corner benchmark.

In [ ]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and run

In [ ]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-2-3-corner.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
result = run_scenario(scenario, seed=42)
walkable = scenario.walkable_polygon
df = result.trajectory_dataframe()

## Boundary-penetration check

In [ ]:
walkable_with_tol = walkable.buffer(0.001)
outside = []
for _, row in df.iterrows():
    if not walkable_with_tol.contains(Point(row.x, row.y)):
        outside.append((int(row.id), int(row.frame), float(row.x), float(row.y)))
print(f'{len(outside)} / {len(df)} trajectory points outside walkable area')

## Plot trajectories

In [ ]:
traj = pedpy.TrajectoryData(
    df[['id', 'frame', 'x', 'y']].copy(),
    frame_rate=result.frame_rate,
)
wa = pedpy.WalkableArea(walkable)
pedpy.plot_trajectories(walkable_area=wa, traj=traj)
plt.show()

## Acceptance

In [ ]:
assert len(outside) == 0, outside[:5]
assert result.agents_remaining == 0, result.agents_remaining
print(f'evacuation time = {result.evacuation_time:.2f} s')
result.cleanup()